In [1]:
import pandas as pd 
import numpy as np

In [4]:
df = pd.read_csv("online_retail_II.csv", encoding="latin1")

In [6]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head(20)

Rows: 541910
Columns: 8


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,12/1/10 8:26,7.65,17850.0,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,12/1/10 8:26,4.25,17850.0,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,12/1/10 8:28,1.85,17850.0,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,12/1/10 8:28,1.85,17850.0,United Kingdom
9,536368,22960,JAM MAKING SET WITH JARS,6,12/1/10 8:34,4.25,13047.0,United Kingdom


In [7]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
Price               0
Customer ID    135080
Country             0
dtype: int64

Step 2: Clean the Transaction Data

In [8]:
# Make a copy
repeat_df = df.copy()

# Remove transactions without Customer ID
repeat_df = repeat_df.dropna(subset=["Customer ID"])

# Remove cancelled invoices
repeat_df = repeat_df[
    ~repeat_df["Invoice"].astype(str).str.startswith("C")
]

# Keep only positive quantities
repeat_df = repeat_df[repeat_df["Quantity"] > 0]

# Convert InvoiceDate to datetime
repeat_df["InvoiceDate"] = pd.to_datetime(repeat_df["InvoiceDate"])

# Convert Customer ID to integer
repeat_df["Customer ID"] = repeat_df["Customer ID"].astype(int)

print("Original rows:", len(df))
print("Cleaned rows:", len(repeat_df))
print("Unique customers:", repeat_df["Customer ID"].nunique())

C:\Users\Bikash Mahato\AppData\Local\Temp\ipykernel_23240\2061573335.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  repeat_df["InvoiceDate"] = pd.to_datetime(repeat_df["InvoiceDate"])


Original rows: 541910
Cleaned rows: 397925
Unique customers: 4339


Step 3: Calculate Orders per Customer

In [9]:
customer_orders = (
    repeat_df.groupby("Customer ID")["Invoice"]
    .nunique()
    .reset_index(name="Number of Orders")
)

customer_orders.head()

,Customer ID,Number of Orders
0,12346,1
1,12347,7
2,12348,4
3,12349,1
4,12350,1


In [10]:
customer_orders.head(10)

,Customer ID,Number of Orders
0,12346,1
1,12347,7
2,12348,4
3,12349,1
4,12350,1
5,12352,8
6,12353,1
7,12354,1
8,12355,1
9,12356,3


In [11]:
customer_orders["Number of Orders"].describe()

count    4339.000000
mean        4.271952
std         7.705493
min         1.000000
25%         1.000000
50%         2.000000
75%         5.000000
max       210.000000
Name: Number of Orders, dtype: float64

Step 4: Identify Repeat Customers

1 order      → One-time Customer

More than 1  → Repeat Customer


In [12]:
customer_orders["Customer Type"] = np.where(
    customer_orders["Number of Orders"] > 1,
    "Repeat Customer",
    "One-time Customer"
)

customer_orders.head(10)

,Customer ID,Number of Orders,Customer Type
0,12346,1,One-time Customer
1,12347,7,Repeat Customer
2,12348,4,Repeat Customer
3,12349,1,One-time Customer
4,12350,1,One-time Customer
5,12352,8,Repeat Customer
6,12353,1,One-time Customer
7,12354,1,One-time Customer
8,12355,1,One-time Customer
9,12356,3,Repeat Customer


In [13]:
customer_orders["Customer Type"].value_counts()

Customer Type
Repeat Customer      2845
One-time Customer    1494
Name: count, dtype: int64

In [14]:
total_customers = customer_orders["Customer ID"].nunique()

repeat_customers = (
    customer_orders["Customer Type"] == "Repeat Customer"
).sum()

repeat_rate = (repeat_customers / total_customers) * 100

print("Total Customers:", total_customers)
print("Repeat Customers:", repeat_customers)
print("Repeat Rate:", round(repeat_rate, 2), "%")

Total Customers: 4339
Repeat Customers: 2845
Repeat Rate: 65.57 %


Step 5: Analyze Repeat vs One-time Customers

Calculate customer spending

In [15]:
customer_spend = (
    repeat_df.groupby("Customer ID")
    .apply(lambda x: (x["Quantity"] * x["Price"]).sum())
    .reset_index(name="Total Spend")
)

C:\Users\Bikash Mahato\AppData\Local\Temp\ipykernel_23240\3425000760.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: (x["Quantity"] * x["Price"]).sum())


In [16]:
customer_analysis = customer_orders.merge(
    customer_spend,
    on="Customer ID",
    how="left"
)

In [17]:
customer_analysis.head(10)

,Customer ID,Number of Orders,Customer Type,Total Spend
0,12346,1,One-time Customer,77183.60
1,12347,7,Repeat Customer,4310.00
2,12348,4,Repeat Customer,1797.24
3,12349,1,One-time Customer,1757.55
4,12350,1,One-time Customer,334.40
5,12352,8,Repeat Customer,2506.04
6,12353,1,One-time Customer,89.00
7,12354,1,One-time Customer,1079.40
8,12355,1,One-time Customer,459.40
9,12356,3,Repeat Customer,2811.43


Compare the two customer groups

In [18]:
customer_analysis.groupby("Customer Type").agg(
    Customers=("Customer ID", "count"),
    Average_Orders=("Number of Orders", "mean"),
    Average_Spend=("Total Spend", "mean"),
    Total_Spend=("Total Spend", "sum")
).round(2)

,Customers,Average_Orders,Average_Spend,Total_Spend
Customer Type,,,,
One-time Customer,1494,1.00,412.52,616311.73
Repeat Customer,2845,5.99,2915.68,8295114.17


Step 6 — Create Customer Segments

In [19]:
customer_analysis["Customer Segment"] = np.select(
    [
        customer_analysis["Number of Orders"] == 1,
        customer_analysis["Number of Orders"].between(2, 3),
        customer_analysis["Number of Orders"].between(4, 10),
        customer_analysis["Number of Orders"] >= 11
    ],
    [
        "One-time",
        "Occasional",
        "Regular",
        "Loyal"
    ],
    default="Other"
)

customer_analysis["Customer Segment"].value_counts()

Customer Segment
One-time      1494
Occasional    1343
Regular       1165
Loyal          337
Name: count, dtype: int64

In [20]:
segment_summary = customer_analysis.groupby("Customer Segment").agg(
    Customers=("Customer ID", "count"),
    Average_Orders=("Number of Orders", "mean"),
    Average_Spend=("Total Spend", "mean"),
    Total_Spend=("Total Spend", "sum")
).round(2)

segment_summary

,Customers,Average_Orders,Average_Spend,Total_Spend
Customer Segment,,,,
Loyal,337,21.12,13029.24,4390855.21
Occasional,1343,2.38,1036.67,1392251.23
One-time,1494,1.00,412.52,616311.73
Regular,1165,5.78,2156.23,2512007.73


Step 7 — Calculate Repeat Rate

In [21]:
total_customers = customer_analysis["Customer ID"].nunique()

repeat_customers = (
    customer_analysis["Number of Orders"] > 1
).sum()

repeat_rate = (repeat_customers / total_customers) * 100

print("Total Customers:", total_customers)
print("Repeat Customers:", repeat_customers)
print("One-time Customers:", total_customers - repeat_customers)
print("Repeat Rate:", round(repeat_rate, 2), "%")

Total Customers: 4339
Repeat Customers: 2845
One-time Customers: 1494
Repeat Rate: 65.57 %


Step 8 — Calculate segment percentages

In [22]:
segment_summary["Customer %"] = (
    segment_summary["Customers"] / total_customers * 100
).round(2)

segment_summary["Spend %"] = (
    segment_summary["Total_Spend"] /
    segment_summary["Total_Spend"].sum() * 100
).round(2)

segment_summary

,Customers,Average_Orders,Average_Spend,Total_Spend,Customer %,Spend %
Customer Segment,,,,,,
Loyal,337,21.12,13029.24,4390855.21,7.77,49.27
Occasional,1343,2.38,1036.67,1392251.23,30.95,15.62
One-time,1494,1.00,412.52,616311.73,34.43,6.92
Regular,1165,5.78,2156.23,2512007.73,26.85,28.19


Step 9 — Final Business Insights

Insight 1 — Repeat Purchase Rate
The customer repeat rate is approximately 65.57%, meaning most identified customers made more than one purchase.

Insight 2 — Loyal Customers Drive Revenue
Loyal customers represent only 7.77% of customers but account for 49.27% of total customer spend.

Insight 3 — One-time Customers Are a Conversion Opportunity
One-time customers are the largest segment at 34.43%, but contribute only 6.92% of total spend. Encouraging a second purchase could potentially increase their customer value.

Insight 4 — Regular Customers Are an Important Middle Segment
Regular customers represent 26.85% of customers and contribute 28.19% of spend. They form an important segment between occasional and loyal customers.

Insight 5 — Purchase Frequency and Customer Value
Average spending rises considerably across the segments:
£412.52 → £1,036.67 → £2,156.23 → £13,029.24

from One-time → Occasional → Regular → Loyal.

In [23]:
customer_analysis.to_csv(
    "Day_28_Customer_Repeat_Purchase_Analysis.csv",
    index=False
)

In [24]:
print(customer_analysis.shape)
print(customer_analysis.columns.tolist())

(4339, 5)
['Customer ID', 'Number of Orders', 'Customer Type', 'Total Spend', 'Customer Segment']
